# SHL Assessment Recommender — Colab Testing

This notebook lets you test the FastAPI agent locally or via ngrok.

In [ ]:
!pip install fastapi uvicorn groq sentence-transformers faiss-cpu pydantic python-dotenv pyngrok requests

In [ ]:
import os
# Set your Groq API key
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'  # Replace with actual key

In [ ]:
# Upload project files or clone from GitHub
# Option 1: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/shl-recommender.git
# %cd shl-recommender

# Option 2: If files are already uploaded, just verify
!ls app/ data/

In [ ]:
import subprocess, time, threading

# Start FastAPI server in background
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(30)  # Wait for model loading + FAISS index build
print('Server should be running on port 8000')

In [ ]:
import requests

# Test health endpoint
resp = requests.get('http://localhost:8000/health')
print(resp.json())

In [ ]:
import requests, json

# Test a simple conversation
payload = {
    'messages': [
        {'role': 'user', 'content': 'I need assessments for hiring Java developers at mid-level.'}
    ]
}
resp = requests.post('http://localhost:8000/chat', json=payload, timeout=30)
data = resp.json()
print('Reply:', data['reply'])
print('Recommendations:', json.dumps(data['recommendations'], indent=2))
print('End:', data['end_of_conversation'])

In [ ]:
# Test vague query (should clarify)
payload = {
    'messages': [
        {'role': 'user', 'content': 'We need a solution for senior leadership.'}
    ]
}
resp = requests.post('http://localhost:8000/chat', json=payload, timeout=30)
data = resp.json()
print('Reply:', data['reply'])
print('Recommendations:', data['recommendations'])
print('End:', data['end_of_conversation'])

In [ ]:
# (Optional) Expose via ngrok for external testing
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f'Public URL: {public_url}')

In [ ]:
# Run evaluation
!python eval/evaluate.py

In [ ]:
# Cleanup
proc.terminate()